In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


##_______________________________________________________________ "" ____________________________________________________________

##_______________________________________________________________ "" ____________________________________________________________

In [5]:
import pandas as pd
import numpy as np

# ============================================================
# DPD30+ — EVENT-LEVEL POINT-IN-TIME BASE
# Cada evento recebe como histórico APENAS o snapshot
# da linha imediatamente anterior do mesmo cliente.
# ============================================================

PATH = r"..\data\processed\features_checkpoint_04F1_TEMP.pkl"

df = pd.read_pickle(PATH).copy()

# ------------------------------------------------------------
# 1. Tipos
# ------------------------------------------------------------

df["sent_at"] = pd.to_datetime(df["sent_at"])

# ------------------------------------------------------------
# 2. Ordenação cronológica
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["customer_id", "sent_at", "event_number"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Sequência real dentro do cliente
# ------------------------------------------------------------

df["event_seq"] = (
    df.groupby("customer_id").cumcount() + 1
)

# ------------------------------------------------------------
# 4. Identificar evento imediatamente anterior
# ------------------------------------------------------------

g = df.groupby("customer_id", sort=False)

df["prev_event_id"] = g["event_id"].shift(1)
df["prev_sent_at"] = g["sent_at"].shift(1)
df["prev_event_number"] = g["event_number"].shift(1)

# ------------------------------------------------------------
# 5. Distância temporal até evento anterior
# ------------------------------------------------------------

df["hours_since_prev_event"] = (
    df["sent_at"] - df["prev_sent_at"]
).dt.total_seconds() / 3600

df["days_since_prev_event"] = (
    df["hours_since_prev_event"] / 24
)

# ------------------------------------------------------------
# 6. Flags
# ------------------------------------------------------------

df["has_prev_event"] = df["prev_event_id"].notna().astype("int8")
df["is_first_event"] = (df["event_seq"] == 1).astype("int8")

# ------------------------------------------------------------
# 7. População do modelo
# IMPORTANTE:
# filtramos DPD30+ SOMENTE depois de construir o lag.
#
# Assim, se o evento atual é DPD31 e o anterior era DPD28,
# ele continua enxergando corretamente o evento DPD28.
# ------------------------------------------------------------

df_30p = (
    df.loc[df["current_dpd"] >= 30]
    .copy()
    .reset_index(drop=True)
)

print("=" * 80)
print("DPD30+ EVENT BASE")
print("=" * 80)

print(f"Rows             : {len(df_30p):,}")
print(f"Customers        : {df_30p['customer_id'].nunique():,}")
print(f"With prev event  : {df_30p['has_prev_event'].sum():,}")
print(f"Without prev     : {(df_30p['has_prev_event'] == 0).sum():,}")

print("\nDPD:")
print(
    df_30p["current_dpd"]
    .describe()
    .round(2)
)

SystemError: deallocated bytearray object has exported buffers

MemoryError: 